<a href="https://colab.research.google.com/github/Kushagra14/triton_tutorials/blob/main/Triton_dropout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dropout needs a random decision for every element

Say your input has 10 numbers, and for each one you flip a coin: keep or drop. Those 10 decisions are the mask:



```
x      = [ 0.5, -1.2, 0.3, 2.0, -0.7, 1.1, 0.9, -0.4, 1.6, 0.2 ]
mask   = [  0,    1,   0,   0,    0,   1,   1,    0,   1,   1  ]   (1 = keep, 0 = drop)
output = x * mask / (1 - p)
```



**Approach 1: store the mask (the baseline)**

You generate those 0s and 1s once, store them in a tensor the same shape as x, and pass it to the kernel. In the tutorial the “bit mask tensor” is this array of 0/1 values. (They stored it as int32, so each 0/1 actually takes 4 bytes, though even at 1 bit per element it would still be one entry per input element.)

If x has 1 billion elements, the mask has 1 billion entries. You also have to keep it around until backward, because backward needs to know which elements were dropped (their gradient is zero too). So memory grows with the tensor size.

**Approach 2: store only a seed (the seeded version)**

A seed is just one int32 number, like 123. 4 bytes total, regardless of tensor size.

The trick is a function that turns (seed, position) into a random number:



```
random(123, 0) = 0.12
random(123, 1) = 0.87
random(123, 2) = 0.33
```


...

It’s deterministic: the same seed and the same position always give the same number. So you don’t need to store the mask. You can recreate it whenever you need it:

```
random = tl.rand(seed, offsets)   # one random number per element, from (seed, position)
x_keep = random > p               # keep if the random number exceeds p
```

Forward pass: call with seed 123, get a mask. Backward pass: call with seed 123 again, get the identical mask. Nothing was stored except the number 123.



**Analogy**

Think of two ways to share a huge list of random numbers with a friend:

Store the mask: send them the whole list.
Seed: send them one number and the recipe, and they regenerate the same list themselves.

The seed is “state” only in this sense: it’s the one thing you need to remember to reproduce all the randomness. In the baseline, the state is the whole mask tensor. Here, it’s one integer.



**Why this isn’t a normal random generator**

A normal generator like torch.rand works sequentially: each number depends on the previous one, so you can’t jump straight to element 1,000,000 without generating everything before it. That’s bad on a GPU where thousands of blocks run at once.

tl.rand uses a counter-based design: the output depends only on (seed, position). Block 500 can compute its own random numbers directly without waiting on blocks 0 to 499.

Does the “same function, same inputs, same output” part make sense now? If so, the next step would be looking at how (seed, offset) actually gets scrambled into a random number.

Short version: a classic generator keeps a hidden state, and every number it gives you changes that state. The next number is computed from it.

**A simple sequential generator (LCG)**

```
state = seed
next():
    state = (a * state + c) mod m
    return state
```

With toy numbers (a=5, c=3, m=16, seed=1):

```
state: 1 -> 8 -> 11 -> 10 -> 5 -> 12 -> ...
```

Each output comes from the previous state. To get the 1,000,000th number, you have to step through the first 999,999 (or use special jump-ahead math). That's the "based on past data" part.

**What torch.rand does**

- **On CPU**, PyTorch uses Mersenne Twister (mt19937). Its state is an array of 624 32-bit integers. Each call reads that state, produces numbers, and updates it. So `torch.rand(3)` called twice gives different results, because the state moved forward in between. `torch.manual_seed(0)` just resets that state.
- **On GPU (CUDA)**, I oversimplified earlier. PyTorch's CUDA generator is also Philox, the same family Triton uses. Its state is just two numbers: a **seed** and an **offset** (a counter). Each time you launch a random kernel, PyTorch hands that kernel a range of counter values and then advances the offset. Thread `i` computes Philox(seed, offset + something based on i), so threads don't depend on each other.

So the "memory of the past" in CUDA PyTorch is a single counter that increments after each random call. That's why:

```python
torch.manual_seed(0)
a = torch.rand(5, device="cuda")   # uses offset 0..
b = torch.rand(5, device="cuda")   # offset has advanced, so different numbers
```

**Where the tutorial differs**

In the tutorial's `seeded_dropout`, you pass the seed in yourself and there's no hidden counter advancing. Same seed means same mask, every call. The state is something you manage explicitly (one int you keep around), instead of PyTorch silently incrementing an offset for you.

**Why this matters for checkpointing**

With PyTorch's global generator, if you recompute a forward pass, the offset has moved on, so you'd get a different mask unless you save and restore the RNG state. That's what `preserve_rng_state` does in checkpointing. With an explicit seed, there's nothing to restore: just reuse the seed.

Does the contrast between "state evolves with each call" and "pure function of (seed, position)" make sense? From here I could show how Philox turns a counter into a random number without any sequential state.

Let me go through it paragraph by paragraph. The last paragraph is worded in a confusing way, so I'll untangle it.

**Paragraph 1: what dropout is for**

A network with lots of parameters and little training data tends to memorize the training set (overfitting). Regularization means techniques that fight this. Dropout is one: during training, randomly switch off some values so the network can't lean too heavily on any single one.

**Paragraph 2: what the operation does**

Input is a vector, output is a vector of the same shape. Each element is independently zeroed with probability `p`, otherwise copied unchanged. With `p = 0.5`:

```
input  = [ 2.0, 4.0, 6.0, 8.0 ]
output = [ 0.0, 4.0, 0.0, 8.0 ]    (random which ones are dropped)
```

"This forces the network to perform well even when only 1 - p of the inputs are available" means: since any value might vanish, the network learns to spread information across many values instead of depending on one. Fewer single points of failure.

**Paragraph 3: the scaling problem**

Dropout is only used in training. At evaluation (inference) you want the full network with nothing dropped, which is the same as `p = 0`.

The issue is magnitude. Take the example above:

```
training: sum of output ≈ half the sum of input (half the values are zeroed)
eval:     nothing dropped, so the sum is the full input
```

So the next layer sees inputs about twice as large at eval as it did during training. The network was trained on smaller-magnitude inputs, so its behavior shifts at eval. The tutorial mentions one example effect: softmax outputs become sharper or flatter than they were in training ("artificial decrease in temperature").

**The fix: multiply by 1/(1-p)**

Scale the *surviving* values up during training:

```
p = 0.5, scale = 1 / (1 - 0.5) = 2

input  = [ 2.0, 4.0, 6.0, 8.0 ]
output = [ 0.0, 8.0, 0.0, 16.0 ]   (survivors doubled)
```

On average, the total is now the same as the input's. Each element is kept with probability `1-p` and scaled by `1/(1-p)`, so its expected value is `(1-p) × x/(1-p) = x`. Training-time outputs therefore match the scale of eval-time outputs, and at eval you just pass the input through untouched.

That's exactly the line in the kernel:

```python
output = tl.where(x_keep, x / (1 - p), 0.0)
```

If kept, divide by `1-p` (same as multiplying by `1/(1-p)`); if dropped, 0.

**The confusing part of the wording**

The text says the multiplication keeps the norm consistent "regardless of the dropout probability", and describes it right after eval. In practice the scaling happens during training (this variant is called "inverted dropout"), so eval needs no special handling. The original 2014 paper did the reverse: no scaling in training, and at eval you scale the weights by `1-p`. Modern frameworks, including PyTorch, use the inverted version, and that's what the tutorial's kernel implements.


In [1]:
import tabulate
import torch

import triton
import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()


@triton.jit
def _dropout(
    x_ptr,  # pointer to the input
    x_keep_ptr,  # pointer to a mask of 0s and 1s
    output_ptr,  # pointer to the output
    n_elements,  # number of elements in the `x` tensor
    p,  # probability that an element of `x` is changed to zero
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    # Load data
    x = tl.load(x_ptr + offsets, mask=mask)
    x_keep = tl.load(x_keep_ptr + offsets, mask=mask)
    # The line below is the crucial part, described in the paragraph above!
    output = tl.where(x_keep, x / (1 - p), 0.0)
    # Write-back output
    tl.store(output_ptr + offsets, output, mask=mask)


def dropout(x, x_keep, p):
    output = torch.empty_like(x)
    assert x.is_contiguous()
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    _dropout[grid](x, x_keep, output, n_elements, p, BLOCK_SIZE=1024)
    return output


# Input tensor
x = torch.randn(size=(10, ), device=DEVICE)
# Dropout mask
p = 0.5
x_keep = (torch.rand(size=(10, ), device=DEVICE) > p).to(torch.int32)
#
output = dropout(x, x_keep=x_keep, p=p)
print(tabulate.tabulate([
    ["input"] + x.tolist(),
    ["keep mask"] + x_keep.tolist(),
    ["output"] + output.tolist(),
]))

---------  --------  --------  --------  --------  ---------  -------  --------  --------  -------  --------
input      0.728186  0.644252  -1.25267  0.140456  -0.735491  1.30656  0.784657  0.157279  1.19978  0.287876
keep mask  1         0          0        0          1         0        0         0         1        1
output     1.45637   0          0        0         -1.47098   0        0         0         2.39956  0.575751
---------  --------  --------  --------  --------  ---------  -------  --------  --------  -------  --------


/usr/local/lib/python3.13/dist-packages/triton/language/semantic.py:1640: UserWarning: tl.where with a non-boolean condition is deprecated and will error out in a future triton release. Got int32
  warnings.warn(


**Seed vs state**

- **Seed**: a number you choose up front. It never changes while you use it. Think of it as the name of one particular random sequence: seed 123 is "sequence #123", seed 512 is "sequence #512".
- **State**: a number (or array) a generator updates each time it produces a value. It records "where we are right now".

In a sequential generator, the seed is only the *initial* state, and then the state moves on:

```
seed = 1
state: 1 -> 8 -> 11 -> 10 -> ...
```

So the seed becomes the state at the start, and the state keeps changing afterward.

**In `tl.rand` there's no evolving state at all**

The seed stays fixed, and the position comes from `offsets`:

```
random = f(seed, offset)
```

Nothing is updated between calls. The seed acts like a **key** that picks one of many possible random lists, and the offset picks which entry of that list you want:

```
seed 123:  [0.12, 0.87, 0.33, 0.51, ...]   <- always this list
seed 512:  [0.64, 0.05, 0.91, 0.22, ...]   <- always this different list
```

**Practical way to think about it**

A seed is like a **book ID** in a library. Book 123 always has the same text. The offset is the page number. You don't need to remember which page you read last, because you can open any page directly. A seed gives you that "which book" choice.

That's also why people call the seed the "state" of low-memory dropout loosely: it's the one thing you must remember to reproduce the mask later. But strictly, it's not changing state, it's a fixed identifier.

----------------------------------------------------------------------
**"Generates a block of uniformly distributed float32 values in [0, 1)"**

*Uniformly distributed* means every value between 0 and 1 is equally likely. No clustering near 0.5, no bias. *[0, 1)* means 0 is possible but 1 is not. *Block* means it produces many numbers at once, one per element your program handles, because Triton kernels work on blocks (vectors) of data rather than single scalars.

**"Given a seed and a block of int32 offsets"**

Those are the two inputs:

```python
random = tl.rand(seed, offsets)
```

- `seed` is a single int32, like 123. It picks *which* random sequence you get.
- `offsets` is a block of int32 positions, like `[0, 1, 2, ..., 1023]` for the first program, `[1024, ..., 2047]` for the second, and so on. It picks *which element* of that sequence you want.

The output is a block of the same length:

```
seed = 123
offsets = [  0,    1,    2,    3  ]
random  = [ 0.12, 0.87, 0.33, 0.51 ]    (made-up numbers)
```

Each output depends on the pair (seed, that offset). Change the seed and you get a different list. Keep the seed and use the same offsets and you get the same list.

**How this connects to dropout**

Then the kernel compares each random number to `p`:

```python
x_keep = random > p
```

With `p = 0.5`, about half the random numbers are above 0.5, so about half the elements are kept. With `p = 0.2`, 80% of the random numbers are above 0.2, so about 80% are kept. That's why you need *uniform* values in [0, 1): the fraction of numbers above `p` is exactly `1 - p`, which gives the right drop rate.

**"Pseudo-random"**

The numbers aren't truly random. They come from a deterministic formula that *looks* random (passes statistical tests) but is fully reproducible. That's what makes seeding useful.

**"Other random number generation strategies"**

`tl.rand` is the simple one. The linked page lists others, such as `tl.randn` for normally distributed (Gaussian) values and `tl.randint` for random integers. Which you'd use depends on what you need: dropout needs a uniform number to compare against `p`, noise injection might want a Gaussian, and so on. They all follow the same pattern of taking a seed and offsets.

In [2]:
@triton.jit
def _seeded_dropout(
    x_ptr,
    output_ptr,
    n_elements,
    p,
    seed,
    BLOCK_SIZE: tl.constexpr,
):
    # compute memory offsets of elements handled by this instance
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    # load data from x
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    # randomly prune it
    random = tl.rand(seed, offsets)
    x_keep = random > p
    # write-back
    output = tl.where(x_keep, x / (1 - p), 0.0)
    tl.store(output_ptr + offsets, output, mask=mask)


def seeded_dropout(x, p, seed):
    output = torch.empty_like(x)
    assert x.is_contiguous()
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    _seeded_dropout[grid](x, output, n_elements, p, seed, BLOCK_SIZE=1024)
    return output


x = torch.randn(size=(10, ), device=DEVICE)
# Compare this to the baseline - dropout mask is never instantiated!
output = seeded_dropout(x, p=0.5, seed=123)
output2 = seeded_dropout(x, p=0.5, seed=123)
output3 = seeded_dropout(x, p=0.5, seed=512)

print(
    tabulate.tabulate([
        ["input"] + x.tolist(),
        ["output (seed = 123)"] + output.tolist(),
        ["output (seed = 123)"] + output2.tolist(),
        ["output (seed = 512)"] + output3.tolist(),
    ]))

-------------------  ---------  -----------  --------  --------  ---------  -------  --------  --------  ---------  -------
input                -0.628392  -0.00565991  0.557244  -1.11248  -0.511822  1.31712  0.304381  0.176949  -0.230813  1.15389
output (seed = 123)   0         -0.0113198   0          0         0         2.63424  0         0         -0.461627  2.30778
output (seed = 123)   0         -0.0113198   0          0         0         2.63424  0         0         -0.461627  2.30778
output (seed = 512)   0          0           1.11449   -2.22496   0         2.63424  0.608762  0          0         0
-------------------  ---------  -----------  --------  --------  ---------  -------  --------  --------  ---------  -------
